
# Food Benchmarking Notebook

This notebook benchmarks recipe classification tasks using the `BenchmarkRunner`. It focuses on difficulty prediction without augmentation and meal type prediction with controlled augmentation strategies.


In [ ]:

import os
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from ml_pipeline.utils.data import (
    load_csv,
    prepare_embeddings_data,
    LabelEncoderHelper,
    filter_meal_types,
)
from ml_pipeline.utils.metrics import METRIC_REGISTRY
from ml_pipeline.utils.visualization import plot_metric_csv
from ml_pipeline.pipelines_torch.models import MODEL_REGISTRY
from ml_pipeline.pipelines_torch.benchmark import BenchmarkRunner
from ml_pipeline.data_augmentation.augmentations import AUGMENTATION_REGISTRY
from ml_pipeline.utils.utils import RESULTS_DIR

EXPERIMENT_PATH_START = "bench_food"
EXPERIMENT_RESULTS_DIR = os.path.join(RESULTS_DIR, EXPERIMENT_PATH_START)
os.makedirs(EXPERIMENT_RESULTS_DIR, exist_ok=True)



## Difficulty Classification
### Data Preparation
Load the recipe dataset, clean the labels, and encode difficulty targets.


In [ ]:

difficulty_df = load_csv("recipes_df.csv")
difficulty_df = difficulty_df.dropna(subset=["embeddings_class", "difficult"])
difficulty_df["difficult"] = difficulty_df["difficult"].replace({"A challenge": "More effort"})

X_difficulty, y_difficulty = prepare_embeddings_data(
    difficulty_df,
    target_column="difficult",
    embedding_column="embeddings_class",
)

difficulty_encoder = LabelEncoderHelper()
difficulty_encoder.fit(y_difficulty)
y_difficulty_encoded = difficulty_encoder.transform(y_difficulty)

difficulty_input_dim = X_difficulty.shape[1]
difficulty_num_classes = len(difficulty_encoder.classes())

print(f"Difficulty classes: {difficulty_encoder.classes()}")
print(f"Feature matrix shape: {X_difficulty.shape}")



### Training and Testing (No Augmentation)
Configure multiple models from `models.py` and benchmark them without applying data augmentation.


In [ ]:

difficulty_metrics = [
    METRIC_REGISTRY["accuracy"],
    METRIC_REGISTRY["f1"],
    METRIC_REGISTRY["precision"],
    METRIC_REGISTRY["recall"],
]

difficulty_model_configs = [
    {
        "name": "difficulty_mlp_classifier",
        "class": MODEL_REGISTRY["mlp_classifier"],
        "params": {"input_dim": difficulty_input_dim, "num_classes": difficulty_num_classes},
    },
    {
        "name": "difficulty_deep_mlp_classifier",
        "class": MODEL_REGISTRY["deep_mlp_classifier"],
        "params": {"input_dim": difficulty_input_dim, "num_classes": difficulty_num_classes},
    },
    {
        "name": "difficulty_random_forest_classifier",
        "class": MODEL_REGISTRY["random_forest_classifier"],
        "params": {"n_estimators": 400, "random_state": 42},
    },
    {
        "name": "difficulty_xgboost_classifier",
        "class": MODEL_REGISTRY["xgboost_classifier"],
        "params": {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "objective": "multi:softprob",
            "num_class": difficulty_num_classes,
            "eval_metric": "mlogloss",
            "reg_lambda": 1.0,
        },
    },
]

difficulty_runner = BenchmarkRunner(
    model_configs=difficulty_model_configs,
    augmentations=[None],
    metrics=difficulty_metrics,
    task_type="classification",
    device="cpu",
    epochs=40,
    batch_size=32,
    early_stopping=10,
    use_class_weights=True,
    path_start=EXPERIMENT_PATH_START,
    use_kfold=True,
    k_folds=5,
    learning_rate=1e-3,
    max_factor=1.0,
    random_state=42,
)

difficulty_runner.run(X_difficulty, y_difficulty_encoded)



### Loss Curves
Plot training and validation loss for each difficulty model.


In [ ]:

for config in difficulty_model_configs:
    model_name = config["name"]
    metrics_file = os.path.join(EXPERIMENT_RESULTS_DIR, f"{model_name}_none_metrics.csv")
    if os.path.exists(metrics_file):
        plot_metric_csv(
            metrics_file,
            metric="loss",
            val_metric="val_loss",
            title=f"{model_name} - Loss Curves",
        )
    else:
        print(f"Metrics file not found for {model_name}: {metrics_file}")



## Meal Type Classification
### Data Preparation
Filter the dataset to the main meal categories and prepare embeddings with encoded targets.


In [ ]:

meal_df = filter_meal_types(load_csv("recipes_df.csv"))
meal_df = meal_df.dropna(subset=["embeddings_class", "meal_type"])

X_meal, y_meal = prepare_embeddings_data(
    meal_df,
    target_column="meal_type",
    embedding_column="embeddings_class",
)

meal_encoder = LabelEncoderHelper()
meal_encoder.fit(y_meal)
y_meal_encoded = meal_encoder.transform(y_meal)

meal_input_dim = X_meal.shape[1]
meal_num_classes = len(meal_encoder.classes())

print(f"Meal type classes: {meal_encoder.classes()}")
print(f"Feature matrix shape: {X_meal.shape}")



### Training with Augmentation
Benchmark multiple models while keeping the existing augmentation strategy for meal type classification.


In [ ]:

def named_augmentation(name):
    base_aug = AUGMENTATION_REGISTRY[name]

    def augmentation(*args, **kwargs):
        return base_aug(*args, **kwargs)

    augmentation.__name__ = name
    return augmentation

meal_metrics = difficulty_metrics
meal_augmentation_names = ["none", "borderline_smote", "smote"]
meal_augmentations = [named_augmentation(name) for name in meal_augmentation_names]

meal_model_configs = [
    {
        "name": "meal_mlp_classifier",
        "class": MODEL_REGISTRY["mlp_classifier"],
        "params": {"input_dim": meal_input_dim, "num_classes": meal_num_classes},
    },
    {
        "name": "meal_deep_mlp_classifier",
        "class": MODEL_REGISTRY["deep_mlp_classifier"],
        "params": {"input_dim": meal_input_dim, "num_classes": meal_num_classes},
    },
    {
        "name": "meal_random_forest_classifier",
        "class": MODEL_REGISTRY["random_forest_classifier"],
        "params": {"n_estimators": 400, "random_state": 42},
    },
]

meal_runner = BenchmarkRunner(
    model_configs=meal_model_configs,
    augmentations=meal_augmentations,
    metrics=meal_metrics,
    task_type="classification",
    device="cpu",
    epochs=40,
    batch_size=32,
    early_stopping=10,
    use_class_weights=True,
    path_start=EXPERIMENT_PATH_START,
    use_kfold=True,
    k_folds=5,
    learning_rate=1e-3,
    max_factor=2.0,
    random_state=42,
)

meal_runner.run(X_meal, y_meal_encoded)



### Loss Curves
Plot training and validation loss for each model/augmentation combination.


In [ ]:

for config in meal_model_configs:
    model_name = config["name"]
    for aug in meal_augmentations:
        metrics_file = os.path.join(EXPERIMENT_RESULTS_DIR, f"{model_name}_{aug.__name__}_metrics.csv")
        if os.path.exists(metrics_file):
            plot_metric_csv(
                metrics_file,
                metric="loss",
                val_metric="val_loss",
                title=f"{model_name} ({aug.__name__}) - Loss Curves",
            )
        else:
            print(f"Metrics file not found for {model_name} with {aug.__name__}: {metrics_file}")



## Next Steps
* Review saved metrics in `results/bench_food` to compare model performance.
* Load the persisted models for downstream evaluation or deployment as needed.
